# Plant Disease AR Diagnoser - SOTA Training Phase

**Architecture:** MobileNetV4-Conv-Large (384x384)


**Goal:** Train the SOTA model and save the PyTorch checkpoint securely.

In [21]:
# Installing ONLY the training libraries. 
# We explicitly cap NumPy to 1.x to ensure perfect compatibility with PyTorch Lightning.
!pip install -q "numpy<2" kaggle lightning torchmetrics timm

## 1. Secure Data Ingestion

In [ ]:
import os

# ⚠️ INSERT YOUR KAGGLE CREDENTIALS HERE ⚠️
os.environ['KAGGLE_USERNAME'] = "qwertyx744" 
os.environ['KAGGLE_KEY'] = "THE API KEY"            

DATA_DIR = "./plant_data"
os.makedirs(DATA_DIR, exist_ok=True)

print("Downloading SOTA Plant Diseases Dataset...")
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p {DATA_DIR} --unzip
print("✅ Dataset downloaded and extracted.")

Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100%|██████████████████████████████████████| 2.70G/2.70G [00:08<00:00, 343MB/s]

✅ Dataset downloaded and extracted.


## 2. Advanced Augmentation Pipeline
Simulating real-world camera conditions (shadows, rotations, lighting changes) so the model doesn't overfit to lab backgrounds.

In [26]:
import os
import lightning as L
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

class PlantDataModule(L.LightningDataModule):
    def __init__(self, data_dir: str, batch_size: int = 32):
        super().__init__()
        
        base_extracted_path = os.path.join(data_dir, "New Plant Diseases Dataset(Augmented)", "New Plant Diseases Dataset(Augmented)")
        if not os.path.exists(base_extracted_path):
            base_extracted_path = os.path.join(data_dir, "new plant diseases dataset(augmented)", "New Plant Diseases Dataset(Augmented)")
            
        self.train_dir = os.path.join(base_extracted_path, "train")
        self.val_dir = os.path.join(base_extracted_path, "valid")
        self.batch_size = batch_size
        
    def setup(self, stage=None):
        train_transforms = transforms.Compose([
            transforms.RandomResizedCrop(384, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)) 
        ])
        
        val_transforms = transforms.Compose([
            transforms.Resize((384, 384)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        print(f"Scanning directories in: {self.train_dir}")
        self.train_ds = ImageFolder(self.train_dir, transform=train_transforms)
        self.val_ds = ImageFolder(self.val_dir, transform=val_transforms)
        
        self.num_classes = len(self.train_ds.classes)
        print(f"✅ Data Pipeline Secure: {self.num_classes} Classes Detected.")

    def train_dataloader(self):
        # 🚀 THE FIX: num_workers=0 forces main-thread loading. No multiprocessing segfaults.
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=0, pin_memory=True)

    def val_dataloader(self):
        # 🚀 THE FIX: num_workers=0
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=0, pin_memory=True)
# 🚀 TEST THE BLUEPRINT
print("Testing the data pipeline...")
test_data = PlantDataModule(data_dir=DATA_DIR)
test_data.setup()

Testing the data pipeline...
Scanning directories in: ./plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
✅ Data Pipeline Secure: 38 Classes Detected.


## 3. MobileNetV4 Engine

In [29]:
import timm
import torch
import torch.nn as nn
import lightning as L
from torchmetrics.classification import MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score

class PlantDiseaseEdgeModel(L.LightningModule):
    def __init__(self, num_classes: int, learning_rate: float = 5e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 🚀 THE FIX: Dropped the brittle 'e500' tag. 
        # Using the base architecture lets timm securely fetch the default pretrained weights.
        self.model = timm.create_model('mobilenetv4_conv_large', pretrained=True, num_classes=num_classes)
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1) 
        
        metric_args = {"num_classes": num_classes, "average": "macro"}
        self.val_acc = MulticlassAccuracy(**metric_args)
        self.val_prec = MulticlassPrecision(**metric_args)
        self.val_rec = MulticlassRecall(**metric_args)
        self.val_f1 = MulticlassF1Score(**metric_args)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log('train_loss', loss, prog_bar=True, on_step=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        
        self.val_acc(preds, y)
        self.val_prec(preds, y)
        self.val_rec(preds, y)
        self.val_f1(preds, y)
        
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_acc', self.val_acc, prog_bar=True, on_epoch=True)
        self.log('val_f1', self.val_f1, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.hparams.learning_rate, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
        return {"optimizer": optimizer, "lr_scheduler": scheduler, "monitor": "val_f1"}

## 4. Run Training Sequence
This will utilize the L4 GPU with 16-bit mixed precision to train rapidly.

In [30]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

# Initialize Pipeline
data_module = PlantDataModule(data_dir=DATA_DIR)
data_module.setup()

model = PlantDiseaseEdgeModel(num_classes=data_module.num_classes)

# Save the absolute best model to a folder named 'checkpoints'
checkpoint_callback = ModelCheckpoint(
    monitor='val_f1',
    mode='max',
    dirpath='checkpoints',
    filename='sota-mnv4-model-{epoch:02d}-{val_f1:.3f}',
    save_top_k=1
)

early_stop_callback = EarlyStopping(
    monitor='val_f1',
    patience=3,
    mode='max'
)

# Start L4 GPU Engine
trainer = L.Trainer(
    max_epochs=12, 
    accelerator='auto', 
    devices=1,
    precision="16-mixed", 
    callbacks=[checkpoint_callback, early_stop_callback]
)

print("🚀 Launching Training Loop...")
trainer.fit(model, datamodule=data_module)
print("✅ Training Complete. Model weights saved securely in the 'checkpoints' folder.")

Scanning directories in: ./plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
✅ Data Pipeline Secure: 38 Classes Detected.


model.safetensors:   0%|          | 0.00/131M [00:00<?, ?B/s]

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


🚀 Launching Training Loop...


You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

Output()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connec
tor.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connec
tor.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=12` reached.


✅ Training Complete. Model weights saved securely in the 'checkpoints' folder.


In [33]:
# 1. Clean and upgrade SciPy alongside LiteRT to fix the internal environment conflict
!pip install -q --upgrade "numpy<2" scipy litert-torch scikit-learn pandas

import os
import glob
import torch
import torch.nn.functional as F
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import litert_torch 

print("🔍 Locating SOTA Checkpoint...")
# Find the most recently saved checkpoint with the highest F1 score
ckpt_files = glob.glob("checkpoints/*.ckpt")
if not ckpt_files:
    raise FileNotFoundError("Could not find any checkpoints in the 'checkpoints' folder. Please check your path.")
    
best_ckpt = max(ckpt_files, key=os.path.getctime)
print(f"Loading weights from: {best_ckpt}")

# Load PyTorch Model
model = PlantDiseaseEdgeModel.load_from_checkpoint(best_ckpt)
model.eval()
model.to('cpu') # Move to CPU to match mobile execution environment

print("\n⚙️ Compiling PyTorch Graph to Native LiteRT (.tflite)...")
# Trace the architecture for a 384x384 mobile camera feed
sample_input = torch.randn(1, 3, 384, 384)

# Run the layout optimization with the freshly repaired SciPy library
edge_model = litert_torch.convert(model.model, (sample_input,))

# Export the final Android asset
tflite_filename = "plant_disease_diagnoser.tflite"
edge_model.export(tflite_filename)
print(f"✅ EXPORT SUCCESS: '{tflite_filename}' is ready for Android Studio!\n")

print("📊 Generating Performance Matrix (Evaluating a validation subset)...")

# Grab a subset of validation data for rapid testing (e.g., 4 batches = 128 images)
val_loader = data_module.val_dataloader()
num_batches_to_test = 4

y_true = []
pt_preds, pt_probs = [], []
tf_preds, tf_probs = [], []

with torch.no_grad():
    for i, (images, labels) in enumerate(val_loader):
        if i >= num_batches_to_test: break
            
        y_true.extend(labels.numpy())
        
        # --- 1. PyTorch Inference ---
        pt_logits = model(images)
        pt_prob = F.softmax(pt_logits, dim=1)
        pt_probs.extend(pt_prob.numpy())
        pt_preds.extend(torch.argmax(pt_prob, dim=1).numpy())
        
        # --- 2. LiteRT (TFLite) Inference ---
        tf_logits = edge_model(images)
        if isinstance(tf_logits, tuple): tf_logits = tf_logits[0] 
        
        tf_prob = F.softmax(tf_logits, dim=1)
        tf_probs.extend(tf_prob.numpy())
        tf_preds.extend(torch.argmax(tf_prob, dim=1).numpy())

# Calculate strict metrics for the subset
def calc_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    return [acc, prec, rec, f1, auc]

pt_metrics = calc_metrics(y_true, pt_preds, pt_probs)
tf_metrics = calc_metrics(y_true, tf_preds, tf_probs)

# Display the Final Table
df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)', 'ROC / AUC (OVR)'],
    'PyTorch (.ckpt)': pt_metrics,
    'TFLite (.tflite)': tf_metrics
})

# Format as percentages for readability
for col in ['PyTorch (.ckpt)', 'TFLite (.tflite)']:
    df[col] = df[col].apply(lambda x: f"{x * 100:.2f}%")

from IPython.display import display, HTML
display(HTML(df.to_html(index=False, classes='table table-striped')))

🔍 Locating SOTA Checkpoint...
Loading weights from: checkpoints/sota-mnv4-model-epoch=11-val_f1=0.995.ckpt

⚙️ Compiling PyTorch Graph to Native LiteRT (.tflite)...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

(00:05) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:04)

...

(00:05) [START] LiteRT-Torch Convert > Run FX Passes

(00:05) [ FAIL] LiteRT-Torch Convert > Run FX Passes

(00:05) [ FAIL] LiteRT-Torch Convert

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3548, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1300/1482388481.py", line 31, in <module>
    edge_model = litert_torch.convert(model.model, (sample_input,))
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/litert_torch/_convert/interface.py", line 323, in convert
    return Converter().convert(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/litert_torch/_convert/interface.py", line 208, in convert
    converted_model = core.convert_signatures(
                      ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/contextlib.py", line 81, in inner
    return func(*args, **kwds)
           ^^^^^^^^^^^^^^^^